# Impute solvents from a processed NPZ

This notebook loads a processed structure `.npz`, optionally strips existing solvents, imputes additional solvents with `Structure.impute_solvents(...)`, and writes the result as an mmCIF file.

Edit the config cell below, then run the notebook top to bottom.

In [ ]:
from pathlib import Path
import sys

BOLTZGEN_SRC = Path("/data/rbg/users/gloriama/dev/foldeverything/src")
NOTEBOOK_ROOT = Path("/data/rbg/users/gloriama/dev/water_conservation")

if str(BOLTZGEN_SRC) not in sys.path:
    sys.path.insert(0, str(BOLTZGEN_SRC))
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from boltzgen.data.data import Structure
from boltzgen.data.write.mmcif import to_mmcif
from gloria_hbond_helpers import (
    gloria_get_solvent_hbond_counts,
    gloria_get_solvent_hbond_counts_and_mask,
    gloria_get_solvent_hbond_mask,
)
from gloria_solvent_filters import (
    gloria_remove_low_b_factor_solvents,
    gloria_remove_weak_solvents,
)
from impute_solvents_with_num_hbonds import impute_solvents_with_num_hbonds


def resolve_npz_path(pdb_id: str, npz_root: Path, npz_path: str | None = None) -> Path:
    if npz_path:
        return Path(npz_path)
    return Path(npz_root) / f"{pdb_id.lower()}.npz"


In [ ]:
# Config
PDB_ID = "1ubq"
NPZ_ROOT = Path("/data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures")
NPZ_PATH = None  # Set to an explicit file path to override PDB_ID + NPZ_ROOT.
OUTPUT_DIR = NOTEBOOK_ROOT / "outputs"

MIN_SOLVENTS = 60
STRIP_SOLVENTS_BEFOREHAND = True
ALLOW_OVERLAP_WITH_REAL_SOLVENTS = False
ONE_SOLVENT_PER_CHAIN = False
INTERACTION_MIN_DIST = 2.5
MAX_SAMPLE_ATTEMPTS = 100


In [ ]:
npz_path = resolve_npz_path(PDB_ID, NPZ_ROOT, NPZ_PATH)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

if not npz_path.exists():
    raise FileNotFoundError(f"Could not find NPZ input: {npz_path}")

structure = Structure.load(npz_path)
loaded_solvents = structure.count_solvents()

if STRIP_SOLVENTS_BEFOREHAND:
    structure = structure.remove_solvents()

post_strip_solvents = structure.count_solvents()
structure = impute_solvents_with_num_hbonds(
    structure,
    one_solvent_per_chain=ONE_SOLVENT_PER_CHAIN,
    min_solvents=MIN_SOLVENTS,
    interaction_min_dist=INTERACTION_MIN_DIST,
    max_sample_attempts=MAX_SAMPLE_ATTEMPTS,
    allow_overlap_with_real_solvents=ALLOW_OVERLAP_WITH_REAL_SOLVENTS,
)
final_solvents = structure.count_solvents()

output_path = output_dir / f"{PDB_ID.lower()}_imputed_first_stripped.cif"
output_path.write_text(to_mmcif(structure))

print(f"Input NPZ: {npz_path}")
print(f"Loaded solvent count: {loaded_solvents}")
print(f"Post-strip solvent count: {post_strip_solvents}")
print(f"Final solvent count: {final_solvents}")
print(f"Wrote CIF to: {output_path}")


In [ ]:
# Example helper calls
example_structure = Structure.load(npz_path)
example_structure = example_structure.to_one_solvent_per_chain(example_structure)

gloria_hbond_counts = gloria_get_solvent_hbond_counts(example_structure)
(
    gloria_hbond_counts_full,
    gloria_present_solvent_chain_mask,
) = gloria_get_solvent_hbond_counts_and_mask(example_structure)
gloria_keep_solvent_mask = gloria_get_solvent_hbond_mask(
    example_structure,
    min_hbonds=2,
)

gloria_weak_filtered_structure = gloria_remove_weak_solvents(
    example_structure,
    min_hbonds=2,
)
gloria_bfactor_filtered_structure = gloria_remove_low_b_factor_solvents(
    example_structure,
    n_keep=10,
)

print(f"Present solvent chains: {gloria_present_solvent_chain_mask.sum()}")
print(f"Per-chain hbond counts shape: {gloria_hbond_counts.shape}")
print(
    "Counts agree across helper variants:",
    (gloria_hbond_counts == gloria_hbond_counts_full).all(),
)
print(f"Chains meeting min_hbonds=2: {gloria_keep_solvent_mask.sum()}")
print(
    f"Solvents after gloria_remove_weak_solvents: {gloria_weak_filtered_structure.count_solvents()}"
)
print(
    f"Solvents after gloria_remove_low_b_factor_solvents(n_keep=10): {gloria_bfactor_filtered_structure.count_solvents()}"
)